In [ ]:
# Reload common
%load_ext autoreload
from common import (
    DATASETS,
    METHODS,
    METRIC_TO_LABEL,
    get_color,
    get_dataset_label,
    get_marker,
    get_method_label,
)

%autoreload 2

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.axes import Axes
from matplotlib.figure import Figure

plt.style.use(["science", "nature"])

In [ ]:
import pandas as pd

width = 397.48 / 72.27
# silver ratio
aspect = 4
height = width / aspect
figsize = (width, height)
x_key = "accuracy_seen_avg"
y_key = "ece_seen_avg"

In [ ]:
from matplotlib.ticker import PercentFormatter


def plot_accuracy_calibration(df: pd.DataFrame, s: float) -> Figure:
    fig, axes = plt.subplots(
        figsize=figsize, ncols=len(DATASETS), gridspec_kw={"wspace": 0.3}
    )
    fig.subplots_adjust(right=0.82)

    for dataset, ax in zip(DATASETS, axes):
        ax: Axes
        dataset_df = df[df["dataset"] == dataset]
        for method in METHODS:
            method_df = dataset_df[dataset_df["method"] == method]

            ax.scatter(
                method_df[x_key],
                method_df[y_key],
                marker=get_marker(method),
                label=get_method_label(method),
                color=get_color(method),
                edgecolors="black",
                s=s,
                linewidths=0.5,
                alpha=0.6,
            )
        ax.set_title(get_dataset_label(dataset))
        ax.xaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))
        ax.yaxis.set_major_formatter(PercentFormatter(xmax=1, decimals=0))

        ax.margins(x=0.1, y=0.1)

    axes[0].set_ylabel(METRIC_TO_LABEL[y_key])
    axes[1].set_xlabel(METRIC_TO_LABEL[x_key])

    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(
        handles,
        labels,
        loc="center left",
        ncol=1,
        bbox_to_anchor=(0.82, 0.5),
        handletextpad=0.1,
    )
    return fig

In [ ]:
df = pd.read_parquet("dataframe/summary.parquet")
plot_accuracy_calibration(df, s=20).savefig(
    "plots/accuracy_calibration.pdf", bbox_inches="tight"
)

In [ ]:
df = pd.read_csv("dataframe/hpsearch.csv")
plot_accuracy_calibration(df, s=10).savefig(
    "plots/hpsearch_plot.pdf", bbox_inches="tight"
)